In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
import pandas as pd

In [63]:
df = pd.read_csv(
    "IMDB Dataset.csv",
    encoding="utf-8",
    on_bad_lines="skip"
)

In [64]:
df.shape

(94495, 2)

In [65]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [66]:
df.isnull().sum()

,0
review,0
sentiment,0


In [67]:
df.drop_duplicates(inplace=True)

In [68]:
df.shape

(49638, 2)

In [69]:
# Pre-Processing

# convert to lowercase
df["review"] = df["review"].str.lower()
# encoding sentiment
# vectorization --- converting the text into numbers


In [70]:
# remove URLs

import re

def remove_url(text):
  re.sub(r"http\S+","",text) #(pattern, replacement, string) \S - remove non white space character
  return text

df["review"] = df["review"].apply(remove_url)

In [71]:
# remove punctuation marks

def remove_punctuations(text):
  text = re.sub(r"[^A-Za-z0-9\s]","",text) #A-Z,A-Z,0-9,\s
  return text

df["review"] = df["review"].apply(remove_punctuations)


In [72]:
# remove html tags

def remove_html(text):
  text = re.sub(r"<.*?>","",text) #A-Z,A-Z,0-9,\s
  return text

df["review"] = df["review"].apply(remove_html)


In [37]:

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [75]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [76]:
# remove stopwords (i.e in, a , is ...general language words

def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word,"")
  return text

df["review"] = df["review"].apply(remove_stopwords)


In [77]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,positive
1,wderful ltle prducti br br filming technique...,positive
2,hugh h werful w pen me h ummer weeken n...,positive
3,bcy fly e boy jke hk zobe cloe pn f...,negative
4,peer me love i vu unng film wch mr mei ...,positive


In [78]:
# Stemming -- convert word into a basic word
# eg. running=> run, played=> play
# PorterStemming

from nltk.stem import PorterStemmer

In [79]:
def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []

  tokens = word_tokenize(text)
  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)
  return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)


In [80]:
df.head()

,review,sentiment
0,e revew nte wtchg 1 oz epo hook rght exctl hpp...,positive
1,wder ltle prducti br br film techniqu unssum l...,positive
2,hugh h wer w pen me h ummer weeken ng n r cne ...,positive
3,bci fli e boy jke hk zobe cloe pn fghg ebr br ...,negative
4,peer me love i vu unng film wch mr mei fer u v...,positive


In [81]:
# Encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [82]:
y = df["sentiment"]

In [83]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
91174,1
91973,1
92766,1
93577,0


In [84]:
# Vectorization --- convert textual datat into numbers

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [86]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3322110 stored elements and shape (49638, 5000)>

In [95]:
## Dataset and DataLoaders

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2, random_state = 42
)

In [96]:
X_test.shape

(9928, 5000)

In [97]:
X_train.shape

(39710, 5000)

In [98]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

X_train = X_train.toarray()
X_test = X_test.toarray()

In [99]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)

In [100]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [102]:
# Build RNN

import torch.nn as nn
import torch.optim as optim

In [103]:
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    #RNNlayer
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

    # fully connected layer
    self.fc = nn.Linear(hidden_size, 1)


  def forward(self, x):
    # optional => shape (num of layers, batch_size, hidden size)
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

    out,_ = self.rnn(x, h0)
    # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
    # 2nd = final hidden state of last timestep

    out = self.fc(out[:, -1, :])
    return out




In [104]:
input_size = X_train.shape[1]

model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [105]:
## Training the RNN

# unsqueeze squeeze

epochs = 10

for epoch in range(epochs):
  model.train()

  for xb,yb in train_loader:
    optimizer.zero_grad()
    xb = xb.unsqueeze(1) # add singleton direction
    outputs = model(xb) # (batch_size, )

    outputs = torch.sigmoid(outputs.squeeze())
    loss = criterion(outputs, yb) # compute loss
    loss.backward() #backprop
    optimizer.step() # weight updates

  print(f" epoch = {epoch +1} / {epochs} and loss = {loss.item()}")

 epoch = 1 / 10 and loss = 0.3214491009712219
 epoch = 2 / 10 and loss = 0.4504936933517456
 epoch = 3 / 10 and loss = 0.4149615168571472
 epoch = 4 / 10 and loss = 0.18253439664840698
 epoch = 5 / 10 and loss = 0.2767435312271118
 epoch = 6 / 10 and loss = 0.32670286297798157
 epoch = 7 / 10 and loss = 0.43373531103134155
 epoch = 8 / 10 and loss = 0.3089107871055603
 epoch = 9 / 10 and loss = 0.404084712266922
 epoch = 10 / 10 and loss = 0.2944605052471161


In [107]:
# evaluation

model.eval()

with torch.no_grad():
  correct_vals = 0
  total_vals = 0

  for xb, yb in test_loader:
    xb = xb.unsqueeze(1)

    outputs = model(xb)
    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    total_vals += yb.size(0)
    correct_vals += (predicted == yb).sum().item()

  print(f"accuracy = {correct_vals/ total_vals*100}")



accuracy = 82.84649476228849
